### Topic 2: Chunk Size & Overlap
### Chunk Size

### Definition

Chunk size refers to the amount of text contained in a single chunk when a document is divided for embedding and retrieval in a Retrieval-Augmented Generation (RAG) system. It determines how much information is represented by a single embedding.

---

### Why is Chunk Size Important?

Chunk size has a direct impact on retrieval quality.

- If the chunk size is too large, a single embedding may represent multiple topics, making retrieval less precise.
- If the chunk size is too small, related information may be split across multiple chunks, causing important context to be lost.

The goal is to choose a chunk size that balances:
- Context preservation
- Retrieval precision

---

### Types of Chunk Size

There are two common approaches:

- Large Chunk Size
- Small Chunk Size

---

### 1. Large Chunk Size

### Definition

A large chunk contains a relatively large amount of text, typically between **800 and 1,200 tokens**. It often includes an entire section or multiple related paragraphs within a single chunk.

### Advantages

- Preserves the complete context of a topic.
- Reduces the likelihood of splitting related information across multiple chunks.
- Produces more complete responses for broad or complex queries.

### Advantage Example

Suppose the following Password Reset FAQ is stored as one large chunk.

```text
Subject: Password Reset

Q: I forgot my password. How do I reset it?

A:
1. Click "Forgot Password."
2. Enter your registered email address.
3. A reset link will be sent to your inbox.
4. The reset link expires after 30 minutes.
5. If you don't receive the email, check your Spam folder.
6. Contact support if the issue persists.
```

**User Query**

> I requested a password reset, but I didn't receive the email. What should I do?

Since the entire FAQ is stored in one chunk, the retriever returns all of the related information together. The language model knows:

- How to request the password reset.
- Where the reset email is sent.
- That the reset link expires after 30 minutes.
- To check the Spam folder if the email is not received.
- To contact support if the problem continues.

Because all related information is available in a single chunk, the model can generate a complete answer.

---

### Disadvantages

- A single chunk may contain multiple topics.
- Embeddings become less specific because they represent several concepts.
- Retrieval may include unnecessary information.

### Disadvantage Example

Suppose multiple FAQs are combined into one large chunk.

```text
Subject: Password Reset
...

----------------------------------

Subject: Email Verification
...

----------------------------------

Subject: Newsletter Subscription
...

----------------------------------

Subject: Change Registered Email
...
```

**User Query**

> How do I verify my email address?

The retriever returns the entire chunk because all four FAQs share the same embedding.

Although the correct answer is present, the retrieved content also contains unrelated information about:

- Password Reset
- Newsletter Subscription
- Changing a Registered Email

As a result, the embedding represents multiple topics instead of one focused topic, reducing retrieval precision.

---

### 2. Small Chunk Size

### Definition

A small chunk contains a relatively small amount of text, typically between **100 and 300 tokens**. Each chunk usually focuses on a single idea or instruction.

### Advantages

- Produces focused embeddings.
- Improves retrieval precision.
- Reduces the amount of irrelevant information retrieved.

### Advantage Example

Instead of storing the entire Password Reset FAQ as one chunk, it is divided into smaller chunks.

**Chunk 1**

```text
Click "Forgot Password."
Enter your registered email address.
```

**Chunk 2**

```text
The reset link expires after 30 minutes.
```

**Chunk 3**

```text
If you don't receive the email, check your Spam folder.
```

**User Query**

> How long is the password reset link valid?

The retriever directly returns Chunk 2.

Since the chunk focuses on a single piece of information, the model retrieves the exact answer without processing unnecessary content.

---

### Disadvantages

- Related information may be split across multiple chunks.
- Important context can be lost.
- Retrieved chunks may produce incomplete answers.

### Disadvantage Example

Suppose an Email FAQ is divided into the following chunks.

**Chunk 1**

```text
Q: Can I change my registered email address?

A: First, verify your identity using the OTP sent to your current email address.
```

**Chunk 2**

```text
After verification, you can update your email address from the Account Settings page.
```

**User Query**

> How do I change my registered email address?

If only Chunk 2 is retrieved, the answer becomes:

> After verification, you can update your email address from the Account Settings page.

The answer is incomplete because it does not explain what needs to be verified first. That information exists in Chunk 1, which was not retrieved.

This demonstrates how very small chunks can lose important context when related information is separated.

---

### Conclusion

There is no universally optimal chunk size.

- A large chunk preserves more context but may reduce retrieval precision because it can contain multiple topics.
- A small chunk improves retrieval precision by focusing on a single topic but increases the risk of losing important context when related information is split across chunk boundaries.

The ideal chunk size depends on the document structure, the nature of the content, and the types of queries the system is expected to answer.
````

### Overlap

### Definition

Overlap is the amount of text that is shared between two consecutive chunks when a document is split. Instead of one chunk ending exactly where the next chunk begins, a portion of the previous chunk is repeated at the beginning of the next chunk.

For example, if a chunk size is **500 tokens** and the overlap is **100 tokens**, the last 100 tokens of Chunk 1 are repeated as the first 100 tokens of Chunk 2.

---

### Why is Overlap Important?

When documents are split into chunks, sentences or ideas often span chunk boundaries. Without overlap, important context may be divided between two chunks, making the retrieved information incomplete.

Overlap helps preserve this context by ensuring that information near a chunk boundary appears in both neighboring chunks.

The goal of overlap is to balance:

- Context preservation across chunk boundaries
- Storage and computation efficiency

---

### Types of Overlap

There are two common approaches:

- No Overlap
- With Overlap

---

### 1. No Overlap

### Definition

In a no-overlap strategy, each chunk starts exactly where the previous chunk ends. No text is repeated between consecutive chunks.

### Advantages

- No duplicated text.
- Requires less storage.
- Faster embedding generation since each token is embedded only once.

### Advantage Example

Suppose an Email FAQ is split as follows.

**Chunk 1**

```text
Q: How do I reset my password?

A:
1. Click "Forgot Password."
2. Enter your registered email address.
```

**Chunk 2**

```text
3. A reset link will be sent to your inbox.
4. The link expires after 30 minutes.
```

Since there is no repeated content, fewer tokens need to be stored and embedded, making the indexing process more efficient.

---

### Disadvantages

- Important context may be split across chunk boundaries.
- Retrieved chunks may become incomplete.
- Retrieval quality may decrease for information located near chunk boundaries.

### Disadvantage Example

Suppose an Email FAQ is split without overlap.

**Chunk 1**

```text
Q: Can I change my registered email address?

A: First, verify your identity using the OTP sent to your current email address.
```

**Chunk 2**

```text
After verification, you can update your email address from the Account Settings page.
```

**User Query**

> How do I change my registered email address?

If only Chunk 2 is retrieved, the answer becomes:

> After verification, you can update your email address from the Account Settings page.

The retrieved chunk does not explain what must be verified first because that information exists only in Chunk 1.

This is a typical chunk-boundary problem caused by having no overlap.

---

### 2. With Overlap

### Definition

In an overlap strategy, a portion of one chunk is repeated at the beginning of the next chunk. This ensures that information near chunk boundaries is available in both chunks.

### Advantages

- Preserves context across chunk boundaries.
- Improves retrieval quality.
- Reduces incomplete answers.
- Increases the likelihood that retrieved chunks contain enough context to be understood independently.

### Advantage Example

Suppose the same Email FAQ is split using overlap.

**Chunk 1**

```text
Q: Can I change my registered email address?

A: First, verify your identity using the OTP sent to your current email address.
```

**Chunk 2**

```text
First, verify your identity using the OTP sent to your current email address.

After verification, you can update your email address from the Account Settings page.
```

Notice that the verification instruction appears in both chunks.

**User Query**

> How do I change my registered email address?

Even if Chunk 2 is retrieved, it already contains the verification step along with the update instructions.

As a result, the model receives enough context to generate a complete answer.

---

### Disadvantages

- Repeated text increases storage requirements.
- More tokens must be embedded, increasing indexing time.
- Excessive overlap can result in redundant retrieval because multiple chunks may contain the same information.

### Disadvantage Example

Suppose each chunk overlaps by 50%.

**Chunk 1**

```text
Click "Forgot Password."
Enter your registered email address.
A reset link will be sent to your inbox.
```

**Chunk 2**

```text
Enter your registered email address.
A reset link will be sent to your inbox.
The reset link expires after 30 minutes.
```

**Chunk 3**

```text
A reset link will be sent to your inbox.
The reset link expires after 30 minutes.
Check your Spam folder if you do not receive the email.
```

The sentence:

> A reset link will be sent to your inbox.

appears in multiple chunks.

As a result:

- More storage is required.
- More embeddings must be generated.
- The retriever may return multiple chunks containing nearly identical information.

---

### Conclusion

Overlap is used to reduce context loss at chunk boundaries.

- Without overlap, storage is efficient, but important context can be lost when information is split across chunks.
- With overlap, retrieval quality improves because neighboring chunks share context, although this comes at the cost of additional storage and computation.

The ideal overlap size depends on the document structure, chunk size, and the amount of context required for accurate retrieval.
```

### Real-World Issues, Edge Cases, Debugging

- **Zero overlap's actual failure mode**: a policy clause like "premature withdrawal incurs a 1% penalty. This does not apply if the FD is closed due to the death of the depositor." — if the chunk boundary happens to land between these two sentences, one chunk says "1% penalty" with no exception mentioned, and the other says "this does not apply" with no idea what "this" refers to. A query about the death-of-depositor exception could retrieve either half and give an incomplete or actively misleading answer.
- **Overlap isn't free**: more overlap means more redundant text re-embedded and re-stored, directly increasing embedding cost and vector store size for the same source corpus. There's a real budget trade-off, not just a quality dial to crank up.
- **Choosing chunk size is itself content-dependent**: a dense policy clause needs a different chunk size than a conversational FAQ answer — a single fixed chunk size across all sources is a simplification, not a guarantee of correctness for every content type.
- **Debugging boundary loss**: if a known fact is in the source document but retrieval consistently misses it, check whether that fact happens to sit exactly at a chunk boundary before assuming the embedding or retrieval ranking is at fault — this is a very common, very specific root cause.

### Design Decisions & Trade-offs

- Overlap measured in sentences vs. characters: sentence-based overlap (carry forward the last N sentences) keeps boundaries coherent the same way sentence-aware chunking does in the first place; character-based overlap is simpler to implement but can re-cut mid-sentence at the new boundary, partially undermining the reason sentence-aware chunking was chosen.
- Fixed overlap value vs. content-aware overlap: a constant number of sentences/characters is simple and predictable; a smarter approach could vary overlap based on detected sentence boundary density or topic shifts, but adds real complexity for a marginal gain at this project's current scale.

### Alternatives & When To Use Each

- Zero overlap — acceptable only when source content has very low information density per sentence and ideas are unlikely to span exactly across a chunk boundary; risky as a default.
- Fixed sentence-count overlap (the fix here) — good general-purpose default, cheap to implement, directly addresses the boundary-cut problem.
- Fixed character-count overlap — simpler to implement than sentence-counting, but can still cut mid-sentence at the new chunk's start.
- Sliding-window overlap with no discrete chunk boundaries at all — maximizes boundary safety, but multiplies storage/embedding cost significantly since nearly all content gets embedded more than once; rarely worth it outside specialized high-recall use cases.

### Common Mistakes & Production Failures

- Shipping a chunker with zero overlap and discovering boundary-cut information loss only after a customer-facing answer misses an exception clause that was technically in the source document.
- Setting overlap too large relative to chunk size, causing massive redundancy where most of each chunk is just a repeat of its neighbor, inflating cost without meaningfully improving retrieval.
- Picking one global chunk size for every source type without checking whether dense policy text and conversational FAQ content actually need different sizes.

### Lead-Level Interview Questions

**Q: Why does zero overlap matter in practice, given that sentence-aware chunking already avoids cutting mid-sentence?**
A: Sentence-aware chunking guarantees individual sentences aren't broken, but says nothing about *related* sentences that happen to straddle a chunk boundary — an exception clause referencing "this" from the sentence before it can still be split across two chunks, each missing half the meaning. Overlap exists specifically to soften that cross-sentence boundary loss, which sentence-awareness alone doesn't solve.

**Q: How would you decide how much overlap to use, rather than picking an arbitrary number?**
A: Tie it to the typical length of a complete idea in your content — if exception clauses or related statements in this corpus typically span two to three sentences, overlap should cover at least that many sentences so a boundary cut never fully separates a complete thought. It should be validated against real retrieval failures (facts known to be in the source but missed at query time), not chosen purely theoretically.

**Q: What's the actual cost of adding overlap, and how would you reason about whether it's worth it?**
A: Overlap means some text gets embedded and stored more than once, increasing embedding calls and vector store size proportionally to the overlap fraction. It's worth it when boundary-cut information loss is a measured, real problem — confirmed by retrieval misses traceable to chunk boundaries — not applied reflexively as a default without checking whether the corpus actually has content dense enough to need it.

### Hidden Concepts Worth Knowing

- **The boundary-cut problem is a specific instance of a general windowing trade-off**: any system that processes a continuous stream in fixed windows (time-series analysis, audio processing, log analysis) faces the same tension — overlap trades redundancy for safety against information that happens to span a window edge.
- **Overlap doesn't fully solve the problem, it just reduces its frequency**: a sufficiently long related passage can still span beyond even a generous overlap window. Overlap shrinks the failure rate, it doesn't eliminate the failure mode entirely — worth stating honestly rather than treating it as a complete fix.

### Revision Summary

> Chunk size and overlap jointly control the coherence-vs-context-loss trade-off at chunk boundaries. This project's `chunk_text()` currently has zero overlap — a real gap where related ideas spanning a chunk boundary can be split with neither resulting chunk containing the complete thought. Adding overlap (carrying forward the last N sentences into the next chunk) directly addresses this, at the cost of some redundant embedding/storage — a trade that should be sized against real content, not picked arbitrarily.

In [1]:
"""
chunker.py
-----------
Sentence-aware chunking with configurable overlap.

Splits a Document's page_content into smaller Documents, never cutting
mid-sentence, and -- unlike a zero-overlap chunker -- carries forward the
last `overlap` sentences of each chunk into the start of the next one, so
an idea spanning a chunk boundary isn't fully lost to one side.
"""

import re
from document import make_document

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text: str) -> list:
    """Naive sentence splitter -- splits after ./!/? followed by whitespace.
    Good enough for structured policy/FAQ text; a real NLP sentence
    tokenizer (e.g. spaCy) would handle abbreviations more robustly."""
    sentences = SENTENCE_SPLIT_RE.split(text.strip())
    return [s.strip() for s in sentences if s.strip()]


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 1) -> list:
    """Sentence-aware chunking with overlap.

    chunk_size : max characters per chunk (soft limit -- a chunk stops
                 adding sentences once it would exceed this length).
    overlap    : number of trailing sentences from the previous chunk to
                 repeat at the start of the next chunk. 0 = no overlap
                 (the old, gap-having behavior).

    IMPORTANT: overlap is automatically capped so it can never consume an
    entire completed chunk. If it did, carrying back "the last N sentences"
    would just rebuild the identical chunk every time and the loop would
    never make progress -- a real infinite-loop bug, not just slowness.
    """
    sentences = split_sentences(text)
    if not sentences:
        return []

    chunks = []
    current = []
    current_len = 0
    i = 0

    while i < len(sentences):
        sentence = sentences[i]
        if current_len + len(sentence) > chunk_size and current:
            chunks.append(" ".join(current))

            # Cap overlap so at least ONE sentence is always dropped --
            # this is what guarantees forward progress.
            safe_overlap = min(overlap, len(current) - 1) if overlap > 0 else 0
            carry_back = current[-safe_overlap:] if safe_overlap > 0 else []

            current = list(carry_back)
            current_len = sum(len(s) for s in current)
            continue  # re-evaluate the same sentence against the new chunk
        current.append(sentence)
        current_len += len(sentence)
        i += 1

    if current:
        chunks.append(" ".join(current))

    return chunks


def chunk_document(doc: dict, chunk_size: int = 500, overlap: int = 1) -> list:
    """Splits one Document into several smaller Documents, preserving and
    extending metadata with a chunk index so each piece is still traceable
    back to its source/page/row."""
    pieces = chunk_text(doc["page_content"], chunk_size=chunk_size, overlap=overlap)
    result = []
    for idx, piece in enumerate(pieces):
        meta = dict(doc["metadata"])
        meta["chunk_index"] = idx
        result.append(make_document(page_content=piece, metadata=meta))
    return result


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty on the applicable rate. "
        "This does not apply if the FD is closed due to the death of the depositor. "
        "In such cases, the full contracted interest rate is paid up to the date of closure. "
        "Senior citizens receive an additional 0.5 percent interest on all tenures. "
        "This additional rate applies only to resident senior citizens aged 60 and above."
    )

    # chunk_size bumped to 180 so each chunk can actually fit 2 sentences --
    # at 120, no chunk could ever hold more than 1 sentence, so overlap had
    # nothing to carry forward and silently did nothing.
    CHUNK_SIZE = 180

    print("--- Zero overlap (the old gap) ---")
    for i, c in enumerate(chunk_text(sample, chunk_size=CHUNK_SIZE, overlap=0)):
        print(f"  Chunk {i}: {c}")

    print("\n--- With overlap=1 (the fix) ---")
    for i, c in enumerate(chunk_text(sample, chunk_size=CHUNK_SIZE, overlap=1)):
        print(f"  Chunk {i}: {c}")

    print("\n--- As Documents (chunk_document) ---")
    doc = make_document(sample, {"source": "fd_policy_demo.txt", "page": 1})
    for d in chunk_document(doc, chunk_size=CHUNK_SIZE, overlap=1):
        print(f"  {d['metadata']}: {d['page_content'][:60]!r}...")

--- Zero overlap (the old gap) ---
  Chunk 0: Premature withdrawal incurs a 1 percent penalty on the applicable rate. This does not apply if the FD is closed due to the death of the depositor.
  Chunk 1: In such cases, the full contracted interest rate is paid up to the date of closure. Senior citizens receive an additional 0.5 percent interest on all tenures.
  Chunk 2: This additional rate applies only to resident senior citizens aged 60 and above.

--- With overlap=1 (the fix) ---
  Chunk 0: Premature withdrawal incurs a 1 percent penalty on the applicable rate. This does not apply if the FD is closed due to the death of the depositor.
  Chunk 1: This does not apply if the FD is closed due to the death of the depositor. In such cases, the full contracted interest rate is paid up to the date of closure.
  Chunk 2: In such cases, the full contracted interest rate is paid up to the date of closure. Senior citizens receive an additional 0.5 percent interest on all tenures.
  Chunk 3: Se